In [ ]:
import time
start_time = time.time()

import pandas as pd
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix


In [ ]:
device = torch.device("mps") if torch.backends.mps.is_available() else torch.device("cpu")
print(f"Selected device: {device}")


In [ ]:
model_name = "textattack/distilbert-base-uncased-MRPC"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)
model.to(device)
model.eval()
id2label = model.config.id2label
print(f"Loaded model: {model_name}")
print(f"Label mapping: {id2label}")


In [ ]:
subset_size = 64
dataset = load_dataset("glue", "mrpc", split=f"validation[:{subset_size}]")
print(f"Validation subset examples: {len(dataset)}")

preview_df = dataset.to_pandas()[["sentence1", "sentence2", "label"]].copy()
print(preview_df.head(5).to_string(index=False))


In [ ]:
batch_size = 16
predictions = []
true_labels = []
positive_class_scores = []

for start_idx in range(0, len(dataset), batch_size):
    batch = dataset[start_idx:start_idx + batch_size]
    inputs = tokenizer(
        batch["sentence1"],
        batch["sentence2"],
        truncation=True,
        padding=True,
        return_tensors="pt"
    )
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)
        probs = torch.softmax(outputs.logits, dim=-1)
        preds = torch.argmax(probs, dim=-1)

    predictions.extend(preds.cpu().tolist())
    true_labels.extend(batch["label"])
    positive_class_scores.extend(probs[:, 1].detach().cpu().tolist())

print(f"Completed inference for {len(predictions)} examples.")


In [ ]:
accuracy = accuracy_score(true_labels, predictions)
precision, recall, f1, _ = precision_recall_fscore_support(
    true_labels,
    predictions,
    average="binary",
    zero_division=0
)
cm = confusion_matrix(true_labels, predictions)

print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1 Score:  {f1:.4f}")
print("Confusion Matrix:")
print(cm)

results_df = pd.DataFrame([
    {
        "model_name": model_name,
        "split": f"validation[:{subset_size}]",
        "num_examples": len(dataset),
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "device": str(device)
    }
])

print(results_df.to_string(index=False))


In [ ]:
examples_df = dataset.to_pandas()[["sentence1", "sentence2", "label"]].copy()
examples_df = examples_df.rename(columns={"label": "true_label"})
examples_df["predicted_label"] = predictions
examples_df["predicted_positive_score"] = positive_class_scores
examples_df["true_label_name"] = examples_df["true_label"].map(id2label)
examples_df["predicted_label_name"] = examples_df["predicted_label"].map(id2label)
examples_df["is_correct"] = examples_df["true_label"] == examples_df["predicted_label"]
examples_df["error_type"] = "correct"
examples_df.loc[(examples_df["true_label"] == 0) & (examples_df["predicted_label"] == 1), "error_type"] = "false_positive"
examples_df.loc[(examples_df["true_label"] == 1) & (examples_df["predicted_label"] == 0), "error_type"] = "false_negative"

columns_to_show = [
    "true_label",
    "predicted_label",
    "true_label_name",
    "predicted_label_name",
    "predicted_positive_score",
    "is_correct",
    "error_type",
    "sentence1",
    "sentence2"
]

print(examples_df[columns_to_show].head(10).to_string(index=False))


In [ ]:
error_summary_df = (
    examples_df.groupby(["true_label", "predicted_label", "error_type"], dropna=False)
    .size()
    .reset_index(name="count")
    .sort_values(["true_label", "predicted_label"])
)
print("Error summary table:")
print(error_summary_df.to_string(index=False))

correct_df = examples_df[examples_df["is_correct"]].copy()
false_positive_df = examples_df[examples_df["error_type"] == "false_positive"].copy()
false_negative_df = examples_df[examples_df["error_type"] == "false_negative"].copy()

print(f"\nCorrect predictions: {len(correct_df)}")
if len(correct_df) > 0:
    print(correct_df[columns_to_show].head(10).to_string(index=False))

print(f"\nFalse positives: {len(false_positive_df)}")
if len(false_positive_df) > 0:
    print(false_positive_df[columns_to_show].to_string(index=False))

print(f"\nFalse negatives: {len(false_negative_df)}")
if len(false_negative_df) > 0:
    print(false_negative_df[columns_to_show].to_string(index=False))


In [ ]:
per_example_table = examples_df[[
    "true_label",
    "predicted_label",
    "predicted_positive_score",
    "is_correct",
    "error_type",
    "sentence1",
    "sentence2"
]].copy()

false_positive_table = false_positive_df[[
    "predicted_positive_score",
    "sentence1",
    "sentence2"
]].sort_values("predicted_positive_score", ascending=False)

false_negative_table = false_negative_df[[
    "predicted_positive_score",
    "sentence1",
    "sentence2"
]].sort_values("predicted_positive_score", ascending=True)

print("Per-example correctness table (first 20 rows):")
print(per_example_table.head(20).to_string(index=False))

print("\nFalse positive table:")
if len(false_positive_table) > 0:
    print(false_positive_table.to_string(index=False))
else:
    print("No false positives in this subset.")

print("\nFalse negative table:")
if len(false_negative_table) > 0:
    print(false_negative_table.to_string(index=False))
else:
    print("No false negatives in this subset.")


In [ ]:
elapsed_seconds = time.time() - start_time
print(f"Total runtime (seconds): {elapsed_seconds:.2f}")
